# K-means en 3D: agrupamiento y tasa de acierto

Este notebook adapta el ejemplo a datos tridimensionales. Cada muestra tiene tres características:

$$
\mathbf{x}_i = [x_{i1}, x_{i2}, x_{i3}] \in \mathbb{R}^3
$$

El objetivo es observar cómo K-means encuentra dos clusters en un espacio de tres dimensiones y cómo se calcula la tasa de acierto cuando se conocen las etiquetas verdaderas de la simulación.

## 1. Idea geométrica en 3D

En dos dimensiones, cada punto se representa en un plano. En tres dimensiones, cada muestra se representa en un espacio con tres ejes.

K-means no cambia su principio: sigue asignando cada punto al centroide más cercano, pero ahora la distancia se calcula en tres dimensiones:

$$
\left\|\mathbf{x}_i - \boldsymbol{\mu}_k\right\|^2 =
(x_{i1}-\mu_{k1})^2 + (x_{i2}-\mu_{k2})^2 + (x_{i3}-\mu_{k3})^2
$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import permutations
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, adjusted_rand_score, confusion_matrix
from mpl_toolkits.mplot3d import Axes3D  # habilita proyección 3D en matplotlib

RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

In [ ]:
def mejor_acierto_por_permutacion(y_true, y_cluster):
    """Calcula el mejor acierto posible cuando los nombres de los clusters son arbitrarios.

    K-means no sabe que una clase debe llamarse 0, 1, 2, etc. Por eso, antes de
    calcular accuracy, se prueban todas las correspondencias posibles entre
    clusters encontrados y etiquetas verdaderas.
    """
    y_true = np.asarray(y_true)
    y_cluster = np.asarray(y_cluster)

    etiquetas_verdaderas = np.unique(y_true)
    etiquetas_cluster = np.unique(y_cluster)

    if len(etiquetas_verdaderas) != len(etiquetas_cluster):
        raise ValueError("El número de etiquetas verdaderas y clusters debe coincidir.")

    mejor_accuracy = -1.0
    mejor_mapeo = None
    mejor_y_pred = None

    for perm in permutations(etiquetas_verdaderas):
        mapeo = {cluster: etiqueta for cluster, etiqueta in zip(etiquetas_cluster, perm)}
        y_pred = np.array([mapeo[c] for c in y_cluster])
        acc = accuracy_score(y_true, y_pred)

        if acc > mejor_accuracy:
            mejor_accuracy = acc
            mejor_mapeo = mapeo
            mejor_y_pred = y_pred

    return mejor_accuracy, mejor_y_pred, mejor_mapeo

## 2. Generación de datos tridimensionales

Se crean dos grupos con medias en tres dimensiones:

$$
\boldsymbol{\mu}_1 = [2,3,4], \qquad
\boldsymbol{\mu}_2 = [6,7,8]
$$

Para simplificar, ambos grupos usan matriz de covarianza identidad:

$$
\Sigma = I_3
$$

In [ ]:
mu = np.array([
    [2, 3, 4],  # Clase 0
    [6, 7, 8],  # Clase 1
])

n_classes = mu.shape[0]
n_points = 100
sigma = np.eye(3)

data_parts = []
true_labels_parts = []

for i in range(n_classes):
    class_data = rng.multivariate_normal(mu[i], sigma, n_points)
    data_parts.append(class_data)
    true_labels_parts.append(np.full(n_points, i, dtype=int))

data = np.vstack(data_parts)
true_labels = np.concatenate(true_labels_parts)

df = pd.DataFrame(data, columns=["dimension_1", "dimension_2", "dimension_3"])
df["etiqueta_verdadera"] = true_labels
df.head()

## 3. Visualización de los datos sin etiquetas

La gráfica 3D permite observar la separación espacial de los puntos. En un problema real, el algoritmo no ve colores ni etiquetas: solo recibe las coordenadas numéricas.

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(data[:, 0], data[:, 1], data[:, 2], s=40, alpha=0.8)
ax.set_title("Datos 3D sin etiquetar")
ax.set_xlabel("Dimensión 1")
ax.set_ylabel("Dimensión 2")
ax.set_zlabel("Dimensión 3")
plt.show()

## 4. Aplicación de K-means en 3D

Se solicita encontrar dos clusters:

$$
K = 2
$$

El método usa las tres características para calcular distancias y actualizar centroides.

In [ ]:
num_clusters = 2

kmeans = KMeans(n_clusters=num_clusters, random_state=RANDOM_STATE, n_init=10)
cluster_idx = kmeans.fit_predict(data)
cluster_centers = kmeans.cluster_centers_

pd.DataFrame(cluster_centers, columns=["centroide_x", "centroide_y", "centroide_z"])

## 5. Visualización de los clusters encontrados

Los puntos se colorean según el cluster asignado por K-means. Los centroides se muestran como estrellas negras.

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(data[:, 0], data[:, 1], data[:, 2], c=cluster_idx, cmap="tab10", s=40, alpha=0.85)
ax.scatter(cluster_centers[:, 0], cluster_centers[:, 1], cluster_centers[:, 2],
           marker="*", s=350, c="black", edgecolor="white", label="Centroides")
ax.set_title("Clasificación K-means en 3D")
ax.set_xlabel("Coordenada X")
ax.set_ylabel("Coordenada Y")
ax.set_zlabel("Coordenada Z")
ax.legend()
plt.show()

## 6. Cálculo de la tasa de acierto

Igual que en 2D, el nombre asignado a cada cluster puede estar intercambiado. Por eso se busca primero la mejor correspondencia entre clusters y etiquetas verdaderas.

La tasa de acierto se calcula como:

$$
\text{tasa de acierto} = \frac{\text{número de muestras correctamente agrupadas}}{\text{número total de muestras}} \times 100
$$

In [ ]:
accuracy, predicted_labels, mapping = mejor_acierto_por_permutacion(true_labels, cluster_idx)
ari = adjusted_rand_score(true_labels, cluster_idx)

print(f"Mejor correspondencia cluster → etiqueta verdadera: {mapping}")
print(f"Tasa de acierto: {accuracy * 100:.2f}%")
print(f"Adjusted Rand Index: {ari:.4f}")

## 7. Matriz de confusión corregida

La matriz de confusión muestra el conteo de aciertos y errores después de corregir la correspondencia entre clusters y clases reales.

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)
cm_df = pd.DataFrame(
    cm,
    index=["Clase real 0", "Clase real 1"],
    columns=["Predicha 0", "Predicha 1"]
)
cm_df

## 8. Actividad propuesta

Acerque las medias de las dos clases modificando la matriz `mu`. También puede aumentar la varianza cambiando `sigma`.

Analice qué ocurre con:

1. la separación visual en 3D;
2. los centroides encontrados;
3. la tasa de acierto;
4. el Adjusted Rand Index.